# Real-Time Web Search with Claude Using free-web-search-ultimate

This cookbook demonstrates how to give Claude real-time web search capabilities using the [free-web-search-ultimate](https://github.com/wd041216-bit/free-web-search-ultimate) Python package — **completely free, no API keys required**.

[![free-web-search-ultimate MCP server](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate/badges/score.svg)](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)

## What you'll learn

- How to use `free-web-search-ultimate` as a Claude tool for real-time web search
- How to build a research assistant that can search the web without any API costs
- How to combine web search results with Claude's reasoning capabilities

## Why free-web-search-ultimate?

| Feature | Details |
|---------|--------|
| **Cost** | Completely free — no API key, no subscription |
| **Privacy** | Uses DuckDuckGo and privacy-respecting search engines |
| **Reliability** | Multiple search backends with automatic fallback |
| **MCP Support** | Works as an MCP server for Claude Desktop and other clients |
| **CLI Support** | Also works as a standalone CLI tool |

## Setup

```bash
pip install free-web-search-ultimate anthropic
```

In [ ]:
# Install required packages
%pip install free-web-search-ultimate anthropic --quiet

In [ ]:
import anthropic
import json
from free_web_search import search, search_news

# Initialize the Anthropic client
client = anthropic.Anthropic()

print("Setup complete!")

## Define the Web Search Tool

We'll define two tools for Claude:
1. `web_search` — General web search
2. `news_search` — Search for recent news

In [ ]:
# Define the tools for Claude
tools = [
    {
        "name": "web_search",
        "description": "Search the web for current information. Use this tool when you need up-to-date information that may not be in your training data, such as recent events, current prices, latest news, or any time-sensitive information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to look up on the web"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to return (default: 5)",
                    "default": 5
                }
            },
            "required": ["query"]
        }
    },
    {
        "name": "news_search",
        "description": "Search for recent news articles. Use this when you need the latest news on a topic.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The news topic to search for"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of news articles to return (default: 5)",
                    "default": 5
                }
            },
            "required": ["query"]
        }
    }
]

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Execute a tool and return the result as a string."""
    try:
        if tool_name == "web_search":
            results = search(
                query=tool_input["query"],
                max_results=tool_input.get("max_results", 5)
            )
        elif tool_name == "news_search":
            results = search_news(
                query=tool_input["query"],
                max_results=tool_input.get("max_results", 5)
            )
        else:
            return f"Unknown tool: {tool_name}"
        
        # Format results as a readable string
        if not results:
            return "No results found."
        
        formatted = []
        for i, result in enumerate(results, 1):
            formatted.append(f"{i}. **{result.get('title', 'No title')}**")
            if result.get('url'):
                formatted.append(f"   URL: {result['url']}")
            if result.get('snippet') or result.get('body'):
                snippet = result.get('snippet') or result.get('body', '')[:300]
                formatted.append(f"   {snippet}")
            formatted.append("")
        
        return "\n".join(formatted)
    except Exception as e:
        return f"Search error: {str(e)}"

print("Tools defined!")

## Build the Research Assistant

Now let's create a function that lets Claude use web search to answer questions.

In [ ]:
def research_assistant(question: str, verbose: bool = True) -> str:
    """A research assistant that uses web search to answer questions."""
    messages = [{"role": "user", "content": question}]
    
    if verbose:
        print(f"Question: {question}")
        print("-" * 60)
    
    while True:
        response = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=4096,
            tools=tools,
            messages=messages,
            system="You are a helpful research assistant with access to real-time web search. When asked about current events, recent developments, or any time-sensitive information, use the web_search or news_search tools to find accurate, up-to-date information. Always cite your sources."
        )
        
        # Check if we need to execute tools
        if response.stop_reason == "tool_use":
            # Process tool calls
            tool_results = []
            
            for block in response.content:
                if block.type == "tool_use":
                    if verbose:
                        print(f"🔍 Searching: {block.input.get('query', '')}")
                    
                    result = execute_tool(block.name, block.input)
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result
                    })
            
            # Add assistant response and tool results to messages
            messages.append({"role": "assistant", "content": response.content})
            messages.append({"role": "user", "content": tool_results})
            
        else:
            # Final response
            final_text = ""
            for block in response.content:
                if hasattr(block, "text"):
                    final_text += block.text
            
            if verbose:
                print(f"\n📝 Answer:\n{final_text}")
            
            return final_text

print("Research assistant ready!")

## Example 1: Current Events

Let's ask Claude about something that requires up-to-date information.

In [ ]:
answer = research_assistant("What are the latest developments in AI in 2025?")

## Example 2: News Search

Now let's search for recent news on a specific topic.

In [ ]:
answer = research_assistant("What's the latest news about open-source AI models?")

## Example 3: Research with Multiple Searches

Claude can perform multiple searches to gather comprehensive information.

In [ ]:
answer = research_assistant(
    "Compare the performance of the top 3 open-source LLMs available today. "
    "Include their benchmark scores and key features."
)

## Using free-web-search as a CLI Tool

The package also works as a standalone CLI tool:

In [ ]:
# You can also use it directly from the command line:
# free-web-search "latest AI news"
# free-web-search --news "OpenAI announcements"

# Or as a Python library:
from free_web_search import search

results = search("Claude AI latest features", max_results=3)
for r in results:
    print(f"Title: {r.get('title')}")
    print(f"URL: {r.get('url')}")
    print()

## Summary

In this cookbook, we've demonstrated how to:

1. **Install** `free-web-search-ultimate` — a free, no-API-key web search library
2. **Define Claude tools** for web search and news search
3. **Build a research assistant** that uses Claude's tool-use capabilities with real-time web search
4. **Handle multi-turn tool use** where Claude performs multiple searches to answer complex questions

### Key Benefits

- **Zero cost**: No API keys or subscriptions needed for web search
- **Privacy-first**: Uses DuckDuckGo and other privacy-respecting search engines
- **Easy integration**: Works with Claude's native tool-use API
- **MCP compatible**: Can also be used as an MCP server for Claude Desktop

### Resources

- [free-web-search-ultimate on GitHub](https://github.com/wd041216-bit/free-web-search-ultimate)
- [free-web-search-ultimate on PyPI](https://pypi.org/project/free-web-search-ultimate/)
- [Glama MCP Server Page](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)
- [Claude Tool Use Documentation](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)